# kv-cache-lab: KV-cache compression on GPU (Colab)

Runs standardized KV-cache compression with [NVIDIA KVPress](https://github.com/NVIDIA/kvpress) and reports the memory-vs-accuracy trade-off at matched compression ratios.

Use a GPU runtime: **Runtime -> Change runtime type -> GPU**. A free T4 handles a 3B model; use an A100 for 7B-8B.

In [ ]:
!nvidia-smi -L

In [ ]:
!pip -q install kvpress transformers accelerate datasets

## Load a model behind the KVPress pipeline
Qwen2.5-3B-Instruct is open and fits a T4. Swap in `Qwen/Qwen3-8B` or `meta-llama/Llama-3.1-8B-Instruct` on a larger GPU (the Llama checkpoints are gated, so run `huggingface-cli login` first).

In [ ]:
import torch
from transformers import pipeline

MODEL = "Qwen/Qwen2.5-3B-Instruct"
pipe = pipeline("kv-press-text-generation", model=MODEL, device_map="cuda", dtype="auto")
print("loaded", MODEL)

## Iso-ratio comparison
Same context and question, several presses, several compression ratios. We record peak GPU memory during generation so the memory saving is measured, not assumed.

In [ ]:
from kvpress import (
    ExpectedAttentionPress,
    SnapKVPress,
    KnormPress,
    ObservedAttentionPress,
    StreamingLLMPress,
)

PRESSES = {
    "expected_attention": ExpectedAttentionPress,
    "snapkv": SnapKVPress,
    "knorm": KnormPress,
    "h2o": ObservedAttentionPress,
    "streaming_llm": StreamingLLMPress,
}
RATIOS = [0.0, 0.25, 0.5, 0.75]

In [ ]:
context = (
    "The following is a technical brief on long-context inference. "
    "The secret passphrase is 'copper lantern seventeen'. " +
    ("Transformers cache key and value vectors per layer and per head to avoid recomputation. "
     "This cache grows linearly with sequence length and dominates memory at long contexts. ") * 200
)
question = "\nWhat is the secret passphrase mentioned near the start?"

rows = []
for name, cls in PRESSES.items():
    for r in RATIOS:
        torch.cuda.reset_peak_memory_stats()
        answer = pipe(context, question=question, press=cls(compression_ratio=r))["answer"]
        peak_gb = torch.cuda.max_memory_allocated() / 1e9
        hit = "copper lantern seventeen" in answer.lower()
        rows.append((name, r, round(peak_gb, 2), hit, answer.strip()[:80]))
        print(f"{name:<20} ratio={r:<4} peak={peak_gb:5.2f}GB  found={hit}")

In [ ]:
import pandas as pd
df = pd.DataFrame(rows, columns=["press", "ratio", "peak_gb", "found_needle", "answer_head"])
df.to_csv("kvpress_isoratio.csv", index=False)
df

## Standardized accuracy on RULER
The numbers above are a quick single-needle probe. For leaderboard-comparable accuracy, run KVPress's own evaluation over RULER. Flags evolve, so read `evaluation/README.md` after cloning.

In [ ]:
!git clone https://github.com/NVIDIA/kvpress.git
!cat kvpress/evaluation/README.md | head -40

## Cross-check against the cost model
Pull in this repo's dependency-free cost model and confirm the measured memory tracks the analytical `2 * layers * kv_heads * head_dim * dtype_bytes * context`.

In [ ]:
# Replace <username> once the repo is pushed to GitHub.
!git clone https://github.com/<username>/kv-cache-lab.git
import sys; sys.path.insert(0, "kv-cache-lab/src")
from kvlab import memory
cfg = memory.MODELS["llama3-8b"]
print("analytical fp16 per token:", memory.human(memory.bytes_per_token(cfg, "fp16")))
print("analytical fp16 at 32K   :", memory.human(memory.cache_bytes(cfg, 32 * 1024, "fp16")))